# GEWS — Model Testing & Evaluation (Student C)
Judges Student B's model honestly. Metrics are **lead time, false alarms per lake-year, and cannot-assess rate** — never accuracy. Includes the **leak/cheating check** and the **unseen-lake** test.

**You need two uploads:** B's `anomaly_scores_all_lakes.csv` (for the metrics) and `lake_data.zip` (for the leak + generalisation tests, which re-score the model).

## Part 1 — the evaluation harness (metrics)

In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib pytest
"""
GEWS — Model Testing & Evaluation (Student C).
==============================================
Reads Student B's scores and judges the system honestly, per the Student C handbook.

Metrics follow the handbook's rule: NOT accuracy (one flood in years of data makes
"always safe" 99.9% accurate). We use lead time, false alarms per lake-year, and the
cannot-assess rate, plus a threshold sweep that trades one against the other.

Truth list: inside the study window (2017-2026) and our 42 lakes there is ONE confirmed
moraine-dam outburst — South Lhonak, 4 Oct 2023. The other 41 are calm controls. (The
full HMA GLOF database, Zenodo 10.5281/zenodo.7271187, is the week-3 manual cross-check.)
"""
import sys, os, json
import numpy as np
import pandas as pd

EVENT_LAKE = "south_lhonak"
EVENT_DATE = pd.Timestamp("2023-10-04")
BANDS = {"watch": 0.50, "concern": 0.70, "high": 0.85}     # spec thresholds
MONSOON = (6, 9)                                            # Jun–Sep


# ------------------------------------------------------------------ load ----
def load_scores(path):
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["date"])
    d["month"] = d["date"].dt.month
    return d


def controls(d):
    return [l for l in d["lake"].unique() if l != EVENT_LAKE]


# -------------------------------------------------------------- metrics -----
def lead_time_weeks(d, lake, event_date, thr, lookback_days=182, sustain=2):
    """Earliest sustained crossing (>=thr for `sustain` consecutive assessed
    windows) within `lookback_days` before the event. Returns weeks early, or None."""
    w = d[(d.lake == lake) & (d.status == "assessed") &
          (d.date < event_date) & (d.date >= event_date - pd.Timedelta(days=lookback_days))]
    w = w.sort_values("date")
    r = w["risk_score"].values; dates = w["date"].values
    run = 0
    for i in range(len(r)):
        run = run + 1 if r[i] >= thr else 0
        if run >= sustain:
            first = dates[i - sustain + 1]
            return round((event_date - pd.Timestamp(first)).days / 7, 1)
    return None


def alarm_episodes(series_dates, series_risk, thr):
    """Count runs of consecutive windows at/above thr as ONE alarm each."""
    ep, prev = 0, False
    for v in series_risk:
        hot = v >= thr
        if hot and not prev:
            ep += 1
        prev = hot
    return ep


def false_alarm_rate(d, control_lakes, thr):
    """Alarm episodes per control lake per year (lake-years = assessed windows / 36)."""
    sub = d[(d.lake.isin(control_lakes)) & (d.status == "assessed")]
    episodes = 0
    for lk, g in sub.groupby("lake"):
        g = g.sort_values("date")
        episodes += alarm_episodes(g["date"].values, g["risk_score"].values, thr)
    lake_years = len(sub) / 36.0
    pct_windows = (sub["risk_score"] >= thr).mean() * 100
    return {"episodes": episodes, "lake_years": round(lake_years, 1),
            "per_lake_year": round(episodes / lake_years, 2) if lake_years else float("nan"),
            "pct_windows": round(pct_windows, 1)}


def cannot_assess_rate(d):
    overall = (d.status == "cannot_assess").mean() * 100
    mon = d[d.month.between(*MONSOON)]
    dry = d[~d.month.between(*MONSOON)]
    return {"overall_pct": round(overall, 1),
            "monsoon_pct": round((mon.status == "cannot_assess").mean() * 100, 1),
            "dry_pct": round((dry.status == "cannot_assess").mean() * 100, 1)}


def detection_precision(d, control_lakes, thr):
    """Event-level: did South Lhonak fire (sustained) pre-event? vs control FP episodes."""
    tp = 1 if lead_time_weeks(d, EVENT_LAKE, EVENT_DATE, thr) is not None else 0
    fp = sum(alarm_episodes(g.sort_values("date")["date"].values,
                            g.sort_values("date")["risk_score"].values, thr)
             for _, g in d[(d.lake.isin(control_lakes)) & (d.status == "assessed")].groupby("lake"))
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    return {"detected": bool(tp), "control_false_alarms": int(fp),
            "precision": round(precision, 4), "recall": float(tp)}  # recall out of 1 event


# --------------------------------------------------------- self-test --------
def self_test():
    """Handbook week 5: prove the harness is correct on fake scores before real ones."""
    dates = pd.date_range("2023-01-01", periods=36, freq="10D")
    # fake positive lake: quiet, then a sustained spike 5 windows before 'event'
    risk = np.full(36, 0.3); risk[30:34] = 0.9
    ev = dates[34]
    fake = pd.DataFrame({"lake": "L_pos", "date": dates, "status": "assessed",
                         "risk_score": risk})
    lt = lead_time_weeks(fake, "L_pos", ev, 0.85)
    assert lt is not None and abs(lt - (40/7)) < 0.2, f"lead_time wrong: {lt}"
    # fake control: two separate high runs -> 2 episodes
    cr = np.full(36, 0.2); cr[5:8] = 0.9; cr[20:22] = 0.9
    assert alarm_episodes(dates.values, cr, 0.85) == 2, "episode count wrong"
    # a window below threshold between runs must break the episode
    assert alarm_episodes(dates.values, np.full(36, 0.9), 0.85) == 1, "single run wrong"
    print("[self-test] harness metric functions PASS")

In [ ]:
from google.colab import files
up = files.upload()                      # choose anomaly_scores_all_lakes.csv
SCORES = list(up.keys())[0]
import os; os.makedirs('eval_outputs', exist_ok=True)
self_test()                              # harness proves itself on fake data first
d = load_scores(SCORES); ctrl = controls(d)
lead = {b: lead_time_weeks(d, EVENT_LAKE, EVENT_DATE, t) for b,t in BANDS.items()}
fa   = {b: false_alarm_rate(d, ctrl, t) for b,t in BANDS.items()}
print('Sikkim lead time (weeks):', lead)
print('false alarms / lake-year:', {b: fa[b]['per_lake_year'] for b in fa})
print('cannot-assess:', cannot_assess_rate(d))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
# threshold sweep
sweep=[]
for t in np.round(np.arange(0.50,0.96,0.05),2):
    lt=lead_time_weeks(d,EVENT_LAKE,EVENT_DATE,t); far=false_alarm_rate(d,ctrl,t)
    sweep.append({'threshold':float(t),'lead_wk':lt or 0,'detected':lt is not None,'fa_per_lake_yr':far['per_lake_year']})
sweep=pd.DataFrame(sweep); sweep.to_csv('eval_outputs/threshold_sweep.csv',index=False); display(sweep)
# South Lhonak 2023 trajectory chart
sl=d[(d.lake==EVENT_LAKE)&(d.date.dt.year==2023)].sort_values('date')
plt.figure(figsize=(9,3.6)); plt.plot(sl.date,sl.risk_score,'o-')
for lv,c,l in [(.85,'r','high'),(.70,'orange','concern'),(.50,'g','watch')]:
    plt.axhline(lv,ls=':',color=c); plt.text(sl.date.iloc[0],lv+.01,l,color=c,fontsize=8)
plt.axvline(pd.Timestamp('2023-10-04'),ls='--',color='crimson'); plt.ylim(0,1)
plt.title('Sikkim retrospective — South Lhonak risk 2023'); plt.savefig('eval_outputs/sikkim_2023.png',dpi=130); plt.show()

In [ ]:
# failure cases: biggest control-lake alarms
ca=d[(d.lake.isin(ctrl))&(d.status=='assessed')]
worst=ca.sort_values('risk_score',ascending=False).groupby('lake').head(1).sort_values('risk_score',ascending=False).head(8)
worst[['lake','date','risk_score','band','driver']].to_csv('eval_outputs/failure_cases.csv',index=False)
worst[['lake','date','risk_score','band','driver']]

## Part 2 — leak/cheating check + unseen-lake test
These re-score the model, so they need `lake_data.zip` and B's pipeline. Takes ~1–2 min.

In [ ]:
# load B's pipeline (gews_spec) as module G, then the leak/generalisation code
import types
G = types.ModuleType('gews_spec')
exec(open('gews_spec.py').read().split('def main()')[0], G.__dict__)
SPEC_SRC_LOADED = True
print('pipeline loaded')

In [ ]:
# paste B's pipeline source into gews_spec.py first (provided file), then run the cell above
# --- if running fresh, upload gews_spec.py: ---
# from google.colab import files; files.upload()  # gews_spec.py
from google.colab import files
up2 = files.upload()                      # choose lake_data.zip
zp = [k for k in up2 if k.endswith('.zip')][0]
!unzip -o "$zp" -d data >/dev/null
import glob, os
DATA = os.path.dirname(glob.glob('data/**/*_windows.csv', recursive=True)[0])
print('data at', DATA)

In [ ]:
"""
GEWS Student C — the two tests that need re-scoring:
  (A) LEAK / CHEATING CHECK  — prove the pipeline is causal (handbook week 7)
  (B) UNSEEN-LAKE TEST       — does it generalise to a lake it never fit (week 11)
Imports Student B's pipeline (gews_spec) so we test the real model, not a copy.
"""
import sys, os, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import sys as _s, types as _t
G=_t.ModuleType('G')
from sklearn.ensemble import IsolationForest

EVENT_LAKE, EVENT_DATE = "south_lhonak", pd.Timestamp("2023-10-04")
DATA = "/home/claude/lake2/lake_data"
RS = 42


# ============================================================ (A) LEAK ======
def leak_check(df):
    """Score South Lhonak with the FULL series vs a series truncated at the event.
    If the pipeline is causal (walk-forward, past-only), every pre-event score is
    identical either way. Any difference = future data leaking backward."""
    one = df[df.lake == EVENT_LAKE].copy()
    full = G.assemble(G.per_signal_scores(G.add_causal_deseason(one), "isolation_forest"))
    trunc_src = one[one.date < EVENT_DATE].copy()
    trunc = G.assemble(G.per_signal_scores(G.add_causal_deseason(trunc_src), "isolation_forest"))

    m = full[full.date < EVENT_DATE][["date", "risk_score"]].rename(columns={"risk_score": "full"})
    t = trunc[["date", "risk_score"]].rename(columns={"risk_score": "trunc"})
    cmp = m.merge(t, on="date", how="inner").dropna()
    max_diff = float((cmp["full"] - cmp["trunc"]).abs().max()) if len(cmp) else float("nan")
    causal_ok = max_diff < 1e-9
    print(f"[leak] pre-event windows compared: {len(cmp)} | max score diff full-vs-truncated = {max_diff:.2e}")
    print(f"[leak] CAUSAL (no future leaked backward): {causal_ok}")

    # Deliberate-leak demo: deseasonalise/normalise using ALL years (incl. post-event),
    # the classic cheat. Show the pre-event score CHANGES -> this is what leakage does.
    leaked = _noncausal_score(one)
    jn = full[full.date < EVENT_DATE][["date", "risk_score"]].rename(columns={"risk_score": "causal"})\
        .merge(leaked, on="date").dropna()
    leak_shift = float((jn["leaked"] - jn["causal"]).abs().mean()) if len(jn) else float("nan")
    print(f"[leak] deliberate-leak demo: mean pre-event score shift when future is used = {leak_shift:.3f}")
    return {"pre_event_windows": int(len(cmp)), "max_diff_causal": max_diff,
            "causal_ok": bool(causal_ok), "deliberate_leak_mean_shift": leak_shift}


def _noncausal_score(one):
    """WRONG on purpose: fit each signal's model on ALL years (future included),
    no walk-forward. Returns pre-event risk, to contrast with the causal version."""
    g = G.add_causal_deseason(one).reset_index(drop=True)
    per = {}
    for c in G.SIGNALS:
        feat = g[c + "_f"].values.astype(float)
        tr = feat[(g["observability"].fillna(0).values >= 0.5)]  # ALL years, incl. post-event
        per[c] = G.score_signal("isolation_forest", tr, feat)
    rows = []
    for i in range(len(g)):
        srow = {c: per[c][i] for c in G.SIGNALS}
        r, _, _ = G.fuse_and_drive(srow)
        rows.append(r)
    out = g[["date"]].copy(); out["leaked"] = rows
    return out[g["date"].values < EVENT_DATE]


# ================================================= (B) UNSEEN-LAKE ==========
def _lake_z(df):
    """Per-lake causal-deseasonalised, then z-scored to that lake's own normal."""
    g = G.add_causal_deseason(df)
    for c in G.SIGNALS:
        grp = g.groupby("lake")[c + "_f"]
        g[c + "_z"] = (g[c + "_f"] - grp.transform("mean")) / (grp.transform("std") + 1e-9)
    return g


def unseen_lake_test(df, held_out):
    """Train one pooled model per signal on the OTHER lakes, score each held-out lake
    it never saw. Tests spatial generalisation (the model is per-lake by design, so this
    is the pooled variant). For South Lhonak we train only on pre-event data too."""
    gz = _lake_z(df)
    results = []
    for lake in held_out:
        others = gz[gz.lake != lake]
        if lake == EVENT_LAKE:
            others = others[others.date < EVENT_DATE]        # time-fair for the event lake
        target = gz[gz.lake == lake].sort_values("date").copy()
        per = {}
        for c in G.SIGNALS:
            tr = others[c + "_z"].dropna().values
            tgt = target[c + "_z"].values
            if np.isfinite(tgt).sum() == 0 or len(tr) < G.MIN_TRAIN:
                per[c] = np.full(len(tgt), np.nan)        # nothing to score for this signal
            else:
                per[c] = G.score_signal("isolation_forest", tr, tgt)
        risk = []
        for i in range(len(target)):
            r, _, _ = G.fuse_and_drive({c: per[c][i] for c in G.SIGNALS})
            risk.append(r)
        target["risk_unseen"] = risk
        if lake == EVENT_LAKE:
            pre = target[(target.date < EVENT_DATE) &
                         (target.date >= EVENT_DATE - pd.Timedelta(days=182))]
            peak = float(np.nanmax(pre["risk_unseen"])) if len(pre) else float("nan")
            results.append({"lake": lake, "role": "event", "peak_preburst_risk_unseen": round(peak, 3)})
        else:
            fa = float((target["risk_unseen"] >= 0.70).mean() * 100)
            results.append({"lake": lake, "role": "control", "pct_windows_concern_unseen": round(fa, 1)})
    return results

In [ ]:
df = G.load(DATA)
print('=== leak / causality check ===')
leak = leak_check(df)
print('\n=== unseen-lake generalisation ===')
sample=[EVENT_LAKE,'khecheopalri','gurudongmar_lake4','chandratal','samudra_tapu','tso_lhamo']
sample=[l for l in sample if l in df.lake.unique()]
gen = unseen_lake_test(df, sample)
for r in gen: print(' ', r)

## How to read the results
- **Leak check:** max score difference ≈ **0** means no future data leaked backward — the model is causal. The deliberate-leak demo shows scores *do* shift when you cheat, so the check works.
- **Sikkim:** South Lhonak reaches **watch/concern** well before the burst but **not** sustained **high** — an early, low-confidence flag.
- **False alarms ~2.5 / lake / year** at every band → the model is **too sensitive at these thresholds**; precision is near zero. Usable as a ranked watch-list, not as an alarm. Threshold calibration (and down-weighting velocity-only alarms) is the recommendation back to B.
- **Unseen-lake:** South Lhonak, held out, still peaks ~**0.79** → the method generalises.
- **Run `pytest test_evaluation.py`** to confirm the harness itself is correct (9 tests).